# Порог для классификатора: 0 или 1?

Нужна загруженная `sample`, колонка `target` — доля использованного лимита.

Два простых артефакта для отчёта:
1. **Таблица** — сколько % клиентов попадают в класс «использовал», если порог разный
2. **Столбчатый график** — то же наглядно
3. **Корзины** — где лежит выборка по уровню утилизации

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# sample = pd.read_parquet("./data/sample.parquet")
y = pd.to_numeric(sample["target"], errors="coerce").dropna()
n = len(y)

# Пороги в понятных процентах (доля лимита → подпись на графике)
THRESHOLDS = [
    (0, "0%"),
    (0.001, "0,1%"),
    (0.01, "1%"),
    (0.02, "2%"),
    (0.05, "5%"),
    (0.10, "10%"),
]

In [ ]:
rows = []
for thr, label in THRESHOLDS:
    share_used = (y > thr).mean()
    rows.append({
        "Порог": label,
        "Считаем «использовал лимит», если утилизация > порога": f"{100 * share_used:.1f}%",
        "Не использовал (класс 0)": f"{100 * (1 - share_used):.1f}%",
        "Класс 1": f"{(y > thr).sum():,}",
    })
display(pd.DataFrame(rows))

print(f"Всего клиентов: {n:,}  |  Ровно 0% утилизации: {100 * (y <= 0).mean():.1f}%")

In [ ]:
labels = [label for _, label in THRESHOLDS]
pct_used = [100 * (y > thr).mean() for thr, _ in THRESHOLDS]
pct_not = [100 - p for p in pct_used]

fig = go.Figure()
fig.add_trace(go.Bar(
    name="Использовал лимит",
    x=labels, y=pct_used,
    text=[f"{p:.1f}%" for p in pct_used],
    textposition="inside",
    marker_color="#2ca02c",
))
fig.add_trace(go.Bar(
    name="Не использовал",
    x=labels, y=pct_not,
    text=[f"{p:.1f}%" for p in pct_not],
    textposition="inside",
    marker_color="#d62728",
))
fig.update_layout(
    barmode="stack",
    title="Как меняется класс 1 / 0 от выбора порога",
    xaxis_title="Минимальная утилизация, чтобы отнести клиента к классу 1",
    yaxis_title="Доля клиентов, %",
    yaxis_range=[0, 100],
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=480,
    width=800,
)
fig.show()

In [ ]:
bins = [
    "0%",
    "0–0,1%",
    "0,1–1%",
    "1–5%",
    "5–10%",
    ">10%",
]
masks = [
    y <= 0,
    (y > 0) & (y <= 0.001),
    (y > 0.001) & (y <= 0.01),
    (y > 0.01) & (y <= 0.05),
    (y > 0.05) & (y <= 0.10),
    y > 0.10,
]
shares = [100 * m.mean() for m in masks]

fig2 = go.Figure(go.Bar(
    x=bins, y=shares,
    text=[f"{s:.1f}%" for s in shares],
    textposition="outside",
    marker_color="#1f77b4",
))
fig2.update_layout(
    title="Где лежат клиенты по фактической утилизации (без выбора порога)",
    xaxis_title="Утилизация лимита",
    yaxis_title="Доля клиентов, %",
    height=420,
    width=800,
)
fig2.show()